# CRM kampaně: od ověřených dat k odpovědi v Genie

Veškerá data jsou syntetická. Výsledky jsou záměrně simulované, včetně budoucích 30/90denních období. Nejde o prognózu ani výsledek skutečné banky. Notebook připraví dvě nové demo datové objekty v odděleném schématu. Nevytváří Genie automaticky.

In [0]:
# Zadej existující katalog, ve kterém máš právo vytvořit schéma a tabulku.
# "workspace" je pouze výchozí návrh; uprav podle svého prostředí.
dbutils.widgets.text("catalog", "workspace", "Existující katalog")
dbutils.widgets.text("schema", "crm_genie_demo", "Nové demo schéma")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
import re
assert all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in [CATALOG, SCHEMA]), "Použij jednoduché identifikátory."
assert SCHEMA.startswith("crm_genie_demo"), "Použij oddělené schéma začínající crm_genie_demo."
FQN = f"{CATALOG}.{SCHEMA}"
print("Cílové schéma:", FQN)


In [0]:
"""Deterministic, fully synthetic CRM campaign fixture. Standard library only."""
import csv, random
from pathlib import Path
SEED = 20261008
SEGMENTS = ['NO_SAVINGS', 'LOW_USAGE', 'GROWTH_POTENTIAL']
def generate_rows():
    rng = random.Random(SEED)
    rows = []
    for seg_idx, segment in enumerate(SEGMENTS):
        # Exact stratified allocation; labels randomly permuted, never assigned by outcome.
        assignments = ['CONTROL'] * 100 + ['A'] * 450 + ['B'] * 450
        rng.shuffle(assignments)
        for i, variant in enumerate(assignments):
            cid = f'SYN_{seg_idx+1}_{i+1:04d}'
            treated = variant != 'CONTROL'
            eligible = rng.random() > .08
            channel = 'NONE' if not treated else ('EMAIL_AND_IB' if eligible else 'IB')
            delivered = treated and eligible and rng.random() < .97
            banner_viewed = treated and rng.random() < .62
            clicked = treated and (delivered or banner_viewed) and rng.random() < (.29 if variant == 'B' else .23)
            started = clicked and segment == 'NO_SAVINGS' and rng.random() < .69
            opened = (started and rng.random() < .70) or (not treated and segment == 'NO_SAVINGS' and rng.random() < .04)
            converted = opened if segment == 'NO_SAVINGS' else (clicked and rng.random() < .58)
            # Existing balances in CZK; all amounts represent average daily balances.
            cur0 = rng.randrange(30_000, 200_001, 100)
            sav0 = 0 if segment == 'NO_SAVINGS' else rng.randrange(0, 15_001, 100) if segment == 'LOW_USAGE' else rng.randrange(150_000, 650_001, 100)
            term0 = rng.choice([0,0,0,100_000,250_000])
            natural = rng.randrange(-25_000, 35_001, 100)
            uplift = 0
            internal = 0
            if treated and converted:
                internal = min(cur0 // 2, rng.choice([10_000,20_000,40_000]))
                if rng.random() < .62:
                    uplift = rng.choice([20_000,50_000,100_000]) * (2 if segment == 'GROWTH_POTENTIAL' else 1)
            # A few designed examples make the semantic trap concrete, separate flag.
            scenario = 'GENERATED'
            if seg_idx == 0 and i == assignments.index('A'):
                variant='A'; treated=True; channel='EMAIL_AND_IB'; delivered=True; banner_viewed=True
                clicked=True; started=True; opened=True; converted=True
                cur0=100_000; sav0=0; term0=0; natural=0; uplift=0; internal=50_000; scenario='INTERNAL_TRANSFER_ONLY'
            if seg_idx == 0 and i == assignments.index('B'):
                variant='B'; treated=True; channel='EMAIL_AND_IB'; delivered=True; banner_viewed=True
                clicked=True; started=True; opened=True; converted=True
                cur0=100_000; sav0=0; term0=0; natural=0; uplift=50_000; internal=0; scenario='NEW_MONEY'
            natural=max(natural, internal-cur0)
            cur30=cur0+natural-internal; sav30=sav0+internal+uplift; term30=term0
            retention=rng.choice([.5,.75,1.0])
            cur90=cur0+int(natural*.8)-internal; sav90=sav0+internal+int(uplift*retention); term90=term0
            banker=treated and started and not opened and rng.random()<.55
            optout=treated and rng.random()<.008
            total0=cur0+sav0+term0; total30=cur30+sav30+term30; total90=cur90+sav90+term90
            rows.append(dict(campaign_id='DEMO_RESERVE_001', client_id=cid, segment=segment,
                variant=variant, is_treatment=int(treated), channel=channel, currency='CZK',
                is_synthetic=1, scenario=scenario, email_delivered=int(delivered), ib_viewed=int(banner_viewed),
                clicked=int(clicked), application_started=int(started), account_opened=int(opened),
                product_action_completed=int(converted), product_funded=int((segment != 'NO_SAVINGS' or opened) and sav30>0),
                banker_task=int(banker), opted_out=int(optout),
                contact_cost_czk=0 if not treated else (2 if delivered else 0)+(1 if banner_viewed else 0)+(180 if banker else 0),
                avg_current_before_czk=cur0, avg_savings_before_czk=sav0, avg_term_before_czk=term0,
                avg_current_d30_czk=cur30, avg_savings_d30_czk=sav30, avg_term_d30_czk=term30,
                avg_current_d90_czk=cur90, avg_savings_d90_czk=sav90, avg_term_d90_czk=term90,
                avg_total_before_czk=total0, avg_total_d30_czk=total30, avg_total_d90_czk=total90,
                deposit_change_d30_czk=total30-total0, deposit_change_d90_czk=total90-total0,
                savings_change_d30_czk=sav30-sav0))
    return rows

def validate(rows):
    assert len(rows)==3000 and len({r['client_id'] for r in rows})==3000
    for r in rows:
        for suffix in ['before','d30','d90']:
            assert r[f'avg_total_{suffix}_czk']==sum(r[f'avg_{p}_{suffix}_czk'] for p in ['current','savings','term'])
        assert r['deposit_change_d30_czk']==r['avg_total_d30_czk']-r['avg_total_before_czk']
        if not r['is_treatment']:
            assert r['variant']=='CONTROL' and r['channel']=='NONE' and r['contact_cost_czk']==0
            assert not any(r[k] for k in ['email_delivered','ib_viewed','clicked','application_started','banker_task','opted_out'])
        assert not r['application_started'] or r['clicked']
        assert r['avg_current_d30_czk']>=0
    example=next(r for r in rows if r['scenario']=='INTERNAL_TRANSFER_ONLY')
    assert example['savings_change_d30_czk']==50000 and example['deposit_change_d30_czk']==0
    return True

def write_csv(path, rows):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)


rows = generate_rows()
validate(rows)
print("Validováno",len(rows),"syntetických klientů")


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
client_table = f"{FQN}.campaign_clients"
metrics_view = f"{FQN}.campaign_metrics"
# Žádný overwrite: notebook nesmí přepsat již existující objekt.
assert not spark.catalog.tableExists(client_table), "Tabulka existuje. Použij nové demo schéma, nebo původní demo objekty odstraň vědomě."
assert not spark.catalog.tableExists(metrics_view), "Pohled existuje. Použij nové demo schéma."
fields = StructType([StructField(k, LongType() if isinstance(v,int) else StringType(), False) for k,v in rows[0].items()])
df = spark.createDataFrame(rows, schema=fields)
df.write.format("delta").mode("ignore").saveAsTable(client_table)
spark.sql(f"COMMENT ON TABLE {client_table} IS 'Syntetický pilot CRM. Jeden řádek = klient v jedné kampani. CZK; zůstatky jsou průměry denních zůstatků. Žádná reálná klientská data.'")


In [0]:
COLUMN_COMMENTS = {'variant': 'Náhodně přiřazená varianta A/B nebo CONTROL uvnitř podsegmentu; hodnoť intention-to-treat.', 'channel': 'Nenáhodně přiřazený komunikační kanál; porovnání kanálů je pouze popisné, nikoli kauzální.', 'avg_total_before_czk': 'Součet průměrných denních běžných, spořicích a termínovaných vkladů; výchozí období D-30 až D-1, CZK.', 'avg_total_d30_czk': 'Součet průměrných denních všech depozit v období D1 až D30, CZK.', 'avg_total_d90_czk': 'Součet průměrných denních všech depozit v období D61 až D90, CZK; není to průměr všech 90 dní.', 'deposit_change_d30_czk': 'Změna celkových depozit D1-D30 proti baseline; není sama inkrementálním efektem kampaně.', 'deposit_change_d90_czk': 'Změna celkových depozit D61-D90 proti baseline.', 'savings_change_d30_czk': 'Změna jen spořicích zůstatků. Nepoužívat jako růst celkových depozit.', 'product_action_completed': 'Produktová akce dle podsegmentu; u NO_SAVINGS založení, u existujících uživatelů simulované využití/navýšení.', 'product_funded': 'Účet má prostředky v D30; u existujících účtů není toto první financování ani nová konverze.', 'contact_cost_czk': 'Pouze modelový náklad kontaktu a úkolu; nezahrnuje úroky, licence, implementaci ani celkové náklady.'}
for column, description in COLUMN_COMMENTS.items():
    safe = description.replace("'", "''")
    spark.sql(f"ALTER TABLE {client_table} ALTER COLUMN {column} COMMENT '{safe}'")


In [0]:
METRICS_SQL = "WITH aggregates AS (\n SELECT segment, variant, COUNT(*) AS client_count,\n AVG(deposit_change_d30_czk) AS avg_change_d30_czk,\n AVG(deposit_change_d90_czk) AS avg_change_d90_czk,\n SUM(deposit_change_d30_czk) AS observed_change_d30_czk,\n SUM(contact_cost_czk) AS contact_cost_czk,\n SUM(product_action_completed) AS completed_actions\n FROM campaign_clients GROUP BY segment, variant\n), control AS (\n SELECT segment, avg_change_d30_czk AS control_avg_d30_czk,\n avg_change_d90_czk AS control_avg_d90_czk\n FROM aggregates WHERE variant='CONTROL'\n)\nSELECT a.*, c.control_avg_d30_czk, c.control_avg_d90_czk,\n a.avg_change_d30_czk-c.control_avg_d30_czk AS incremental_per_client_d30_czk,\n a.avg_change_d90_czk-c.control_avg_d90_czk AS incremental_per_client_d90_czk,\n (a.avg_change_d30_czk-c.control_avg_d30_czk)*a.client_count AS incremental_total_d30_czk,\n (a.avg_change_d90_czk-c.control_avg_d90_czk)*a.client_count AS incremental_total_d90_czk\nFROM aggregates a JOIN control c ON a.segment=c.segment\n"
spark.sql("CREATE VIEW campaign_metrics AS " + METRICS_SQL)
spark.sql("COMMENT ON TABLE campaign_metrics IS 'Agregace segment x experimentální varianta. Inkrementální průměr = změna varianty minus změna kontroly téhož segmentu. Inkrementální celkem = tento rozdíl krát počet přiřazených klientů. Pro celkový výsledek sečti součty A/B; nepočítej prostý průměr průměrů. Syntetická data.'")


In [0]:
assert spark.table(client_table).count() == 3000
assert spark.table(metrics_view).count() == 9
bad = spark.sql("SELECT COUNT(*) AS n FROM campaign_clients WHERE avg_total_d30_czk <> avg_current_d30_czk + avg_savings_d30_czk + avg_term_d30_czk").first().n
assert bad == 0
print("Kontroly prošly. Pro Genie přidej:", client_table, "a", metrics_view)
display(spark.table(metrics_view).orderBy("segment", "variant"))


## Referenční otázky
Spusť dotazy níže a porovnej s `results/expected_answers.md`. Potom stejné otázky polož Genie a zkontroluj také SQL, zdroje a význam metrik.

### Jaký je přírůstek celkových depozit oproti kontrolní skupině po 30 dnech?

In [0]:
display(spark.sql("SELECT ROUND(SUM(incremental_total_d30_czk),2) AS incremental_total_d30_czk,\n ROUND(SUM(incremental_total_d30_czk)/SUM(client_count),2) AS incremental_per_treated_client_d30_czk,\n SUM(client_count) AS treated_clients\nFROM campaign_metrics WHERE variant IN ('A','B')"))


### Který podsegment měl největší přírůstek depozit na osloveného klienta?

In [0]:
display(spark.sql("SELECT segment, SUM(client_count) AS treated_clients,\n ROUND(SUM(incremental_total_d30_czk)/SUM(client_count),2) AS incremental_per_client_d30_czk,\n ROUND(SUM(incremental_total_d30_czk),2) AS incremental_total_d30_czk\nFROM campaign_metrics WHERE variant IN ('A','B') GROUP BY segment\nORDER BY incremental_per_client_d30_czk DESC"))


### Kolik oslovených klientů zvýšilo zůstatek spořicího účtu, ale nezvýšilo celková depozita?

In [0]:
display(spark.sql('SELECT COUNT(*) AS clients_with_savings_growth_without_total_growth\nFROM campaign_clients WHERE is_treatment=1 AND savings_change_d30_czk>0 AND deposit_change_d30_czk<=0'))


### Jak dopadly varianty A a B v jednotlivých podsegmentech?

In [0]:
display(spark.sql("SELECT segment, variant, client_count,\n ROUND(incremental_per_client_d30_czk,2) AS incremental_per_client_d30_czk,\n completed_actions, ROUND(100.0*completed_actions/client_count,2) AS product_action_rate_pct\nFROM campaign_metrics WHERE variant IN ('A','B') ORDER BY segment,variant"))


### Jaký je přírůstek oproti kontrole v období 61 až 90 dní?

In [0]:
display(spark.sql("SELECT ROUND(SUM(incremental_total_d90_czk),2) AS incremental_total_d90_czk,\n ROUND(SUM(incremental_total_d90_czk)/SUM(client_count),2) AS incremental_per_client_d90_czk\nFROM campaign_metrics WHERE variant IN ('A','B')"))


### U klientů bez spořicího produktu ukaž funnel variant A a B.

In [0]:
display(spark.sql("SELECT variant, COUNT(*) AS assigned_clients,\n SUM(clicked) AS clicked_clients, SUM(application_started) AS started_clients,\n SUM(account_opened) AS opened_accounts, SUM(product_funded) AS funded_accounts\nFROM campaign_clients WHERE segment='NO_SAVINGS' AND is_treatment=1 GROUP BY variant ORDER BY variant"))


### Ukaž příklad interního přesunu a příklad nových prostředků.

In [0]:
display(spark.sql("SELECT client_id,scenario,avg_current_before_czk,avg_current_d30_czk,\n avg_savings_before_czk,avg_savings_d30_czk,savings_change_d30_czk,deposit_change_d30_czk\nFROM campaign_clients WHERE scenario<>'GENERATED' ORDER BY scenario"))


### Popisně porovnej výsledky klientů podle komunikačního kanálu.

In [0]:
display(spark.sql('SELECT channel,COUNT(*) AS assigned_clients,\n ROUND(AVG(deposit_change_d30_czk),2) AS avg_observed_change_d30_czk,\n ROUND(100.0*SUM(clicked)/COUNT(*),2) AS click_rate_per_assigned_pct\nFROM campaign_clients WHERE is_treatment=1 GROUP BY channel ORDER BY channel'))


## Další krok
Vytvoř Genie nad oběma objekty, vlož instrukce z `genie/instructions.md` a SQL příklady z `sql/`. Podrobný postup je v `docs/QUICKSTART.md`. Notebook nic neodesílá klientům a nenastavuje přístupy ani veřejné sdílení.